# NNUE Chess Engine Training Notebook

This notebook allows you to train your NNUE model interactively with live loss visualization.

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
from model import ChessNN, fen_to_tensor # Assumes model.py is in the same folder

## 2. Dataset Definition
Same logic as `train.py` but defined here for easy modification.

In [ ]:
class ChessDataset(Dataset):
    def __init__(self, data_path, device='cpu'):
        print(f"Loading data from {data_path}...")
        raw_data = torch.load(data_path)
        self.data = raw_data
        self.device = device
        print(f"Loaded {len(self.data)} positions.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        fen, score = self.data[idx]
        x = fen_to_tensor(fen, device='cpu')
        
        if " b " in fen:
             score = -score

        score = max(-2000, min(2000, score))
        norm_score = score / 100.0
        
        y = torch.tensor([norm_score], dtype=torch.float32)
        return x, y

## 3. Configuration

In [ ]:
DATA_PATH = "dataset.pt"
CHECKPOINT_DIR = "checkpoints_notebook"
BATCH_SIZE = 64
LR = 0.001
EPOCHS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

## 4. Initialize Data and Model

In [ ]:
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

# Load Data
dataset = ChessDataset(DATA_PATH, device=DEVICE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Init Model
model = ChessNN().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

## 5. Training Loop with Live Plot

In [ ]:
losses = []
avg_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for i, (x, y) in enumerate(pbar):
        x, y = x.to(DEVICE), y.to(DEVICE)
        
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        losses.append(loss.item())
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        # Plot every 100 steps
        if i % 100 == 0 and i > 0:
             # Minimal overhead plotting
             pass 

    # End of Epoch
    avg_loss = total_loss / len(dataloader)
    avg_losses.append(avg_loss)
    
    # Live Plot
    clear_output(wait=True)
    plt.figure(figsize=(10, 5))
    plt.plot(avg_losses, label='Epoch Avg Loss')
    plt.title(f"Training Loss (Epoch {epoch+1})")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    print(f"Epoch {epoch+1} done. Avg Loss: {avg_loss:.4f}")
    
    # Save Checkpoint
    path = os.path.join(CHECKPOINT_DIR, "checkpoint_latest.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, path)
    print(f"Saved checkpoint to {path}")